In [ ]:
import sys
import os
import random
import torch
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as transforms
from matplotlib import pyplot as plt
from PIL import Image
from pathlib import Path
from segment_anything import sam_model_registry

In [ ]:
sys.path.append(os.path.abspath(".."))

from sam_lora import SamLoRA
from Util.env_utils import load_segmentation_env, load_as

In [ ]:
load_segmentation_env()

SEED = load_as("SEED", int, 42)

MODEL_OUT_DIR = os.getenv("MODEL_OUT_DIR")
MODEL_CHECKPOINTS_DIR = os.getenv("MODEL_CHECKPOINTS_DIR")
DATASETS_DIR = os.getenv("DATASETS_DIR")

DATASET_NAME = os.getenv("DATASET_NAME", "SAM_LoRA_Augmented")
CROPPED_AUG_IMG_DIR_NAME = os.getenv("CROPPED_AUG_IMG_DIR_NAME", "images_256")
CROPPED_AUG_MASK_DIR_NAME = os.getenv("CROPPED_AUG_MASK_DIR_NAME", "masks_256")

SAM_LORA_MODEL_TYPE = os.getenv("SAM_LORA_MODEL_TYPE", "vit_b")
SAM_LORA_MODEL_CHECKPOINT = os.getenv(
    "SAM_LORA_MODEL_CHECKPOINT", "sam_vit_b_01ec64")
SAM_LORA_RANK = load_as("SAM_LORA_RANK", int, 4)

if MODEL_OUT_DIR is None or MODEL_CHECKPOINTS_DIR is None or DATASETS_DIR is None:
    raise ValueError(
        "MODEL_OUT_DIR, MODEL_CHECKPOINTS_DIR, or DATASETS_DIR environment variable not set.")

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
print("cuda available: ", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_name = "sam_lora_finetuned_params_20251202_175215.pt"
params = torch.load(os.path.join(MODEL_OUT_DIR, model_name))

print(f"Loaded {len(params)} named parameters and {sum(p.numel() for p in params.values())} total parameters from finetuned model")
for name, param in params.items():
    print(f"\n--- {name} --- ")
    print(f"shape: {param.shape}")
    print(
        f"max: {param.max().item()}, min: {param.min().item()}, std: {param.std().item()}")

In [ ]:
FINETUNE_IMAGE_ENCODER = False
FINETUNE_MASK_DECODER = True
FINETUNE_PROMPT_ENCODER = True

sam_checkpoint = str(Path(MODEL_CHECKPOINTS_DIR) /
                     f"{SAM_LORA_MODEL_CHECKPOINT}.pth")
model_type = SAM_LORA_MODEL_TYPE
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)

sam_lora = SamLoRA(sam, r=SAM_LORA_RANK, finetune_img_encoder=FINETUNE_IMAGE_ENCODER,
                   finetune_mask_decoder=FINETUNE_MASK_DECODER, finetune_prompt_encoder=FINETUNE_PROMPT_ENCODER)
sam_lora.to(device)

sam_lora.load_state_dict(params, strict=False)

sam_lora.eval()
print()

In [ ]:
def load_transform_image(path, resize=(1024, 1024)):
    transform = transforms.Compose([transforms.Resize(resize, interpolation=transforms.InterpolationMode.BILINEAR),
                                    transforms.ToTensor(),
                                    transforms.Lambda(lambda t: t.repeat(3, 1, 1) if t.shape[0] == 1 else t)])

    img = Image.open(path)
    print(f"Input image size before transform: {img.size}")
    return transform(img)


def show_mask(mask, ax):
    color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def get_batched_input_list(image: torch.Tensor):
    return [{
        "image": img,
        "original_size": (img.shape[1], img.shape[2])
    } for img in image.unsqueeze(0).unbind(0)]


def evaluate_on_image(image_path, figsize=(10, 10)):
    input_img = load_transform_image(image_path).to(device)
    print(f"Input image shape: {input_img.shape}")

    input_list = get_batched_input_list(input_img)

    outputs = sam_lora(batched_input=input_list, multimask_output=False)
    mask = outputs[0]['masks'].squeeze(0).cpu().numpy()
    print(f"Output mask shape: {mask.shape}")
    print(f"Output mask dtype: {mask.dtype}")

    plt.figure(figsize=figsize)
    plt.imshow(input_img.permute(1, 2, 0).cpu().numpy())
    show_mask(mask, plt.gca())
    plt.axis('off')
    plt.show()

    return mask

In [ ]:
test_img_dir = Path(DATASETS_DIR) / DATASET_NAME / \
    "test" / CROPPED_AUG_IMG_DIR_NAME
random_test_images = random.sample(list(test_img_dir.glob("*.png")), 10)

for img_path in random_test_images:
    evaluate_on_image(str(img_path), figsize=(5, 5))

In [ ]:
'''
lowres_size = (512, 512)


def load_transform_lowres_mask(path):
    transform = transforms.Compose([transforms.Resize((1024, 1024), interpolation=transforms.InterpolationMode.BILINEAR),
                                    transforms.ToTensor()])
    img = Image.open(path)
    img = transform(img)

    img = img.to(dtype=torch.float32).unsqueeze(0)
    img = F.interpolate(img, size=lowres_size,
                        mode='bilinear', align_corners=False)
    return img.squeeze(0).cpu().numpy()


def plot_lowres_groundtruth(image_path):
    input_img = load_transform_image(image_path, resize=lowres_size)
    print(f"Input image shape: {input_img.shape}")

    mask = load_transform_lowres_mask(image_path.replace("images", "masks"))

    plt.figure(figsize=(10, 10))
    plt.imshow(input_img.permute(1, 2, 0).cpu().numpy())
    show_mask(mask, plt.gca())
    plt.axis('off')
    plt.show()


plot_lowres_groundtruth(
    "/data/jhehli/datasets/SAM_LoRA_Augmented/test/images/20250811_dani_vegetariana_soi_2_bottom_rot270.png")
'''